In [4]:
import numpy as np
import pandas as pd

# データ読み込み（3段階ヘッダー、先頭列をインデックスに）
filename = r"F:\thesis\0616SUNNING-akane-2026-06-16\videos\ワオキツネザルCAM1-20250427-085828-1745711908801-7_00-18-50_00-19-20DLC_resnet50_0616SUNNINGJun16shuffle1_100000.csv"
df = pd.read_csv(filename, header=[0, 1, 2], index_col=0)

# 最上位階層（スコアラー名）の文字列を自動抽出
scorer = df.columns[0][0]

# 尤度フィルタ（全部位 >= 0.9 を有効とする）
parts = ["left_hand", "right_hand", "left_ear", "right_ear"]
valid_mask = True
for part in parts:
    valid_mask &= df[(scorer, part, "likelihood")] >= 0.9

# left_hand の座標抽出
hand_x = df[(scorer, "left_hand", "x")]
hand_y = df[(scorer, "left_hand", "y")]

# ローリング標準偏差で Motion Index を計算
window_size = 11  # 前後5フレーム分を含めた窓
std_x = hand_x.rolling(window=window_size, center=True).std()
std_y = hand_y.rolling(window=window_size, center=True).std()
motion_index = np.sqrt(std_x**2 + std_y**2)

# サニング判定（Motion Index < threshold かつ尤度OK）
threshold_px = 2.0
is_sunning = (motion_index < threshold_px) & valid_mask

# サニングフレームのリスト化
sunning_frames = is_sunning[is_sunning].index.tolist()

print(f"スコアラー: {scorer}")
print(f"サニング検知フレーム数: {len(sunning_frames)}")

抽出されたスコアラー名: DLC_resnet50_0616SUNNINGJun16shuffle1_100000
サニング検知フレーム数: 143
